# Comparação de dois modelos gratuitos pelo OpenRouter

Atividade da Aula 09.1 — Gateway Multi-Modelo.

**Objetivo:** executar o mesmo prompt em dois modelos gratuitos e comparar qualidade, latência e limitações com evidências verificáveis. Este arquivo contém o procedimento completo. Nomes e RMs foram omitidos para cumprir a orientação de não enviar dados pessoais.

O OpenRouter encaminha as solicitações aos provedores. A qualidade observada corresponde à combinação de modelo, provedor e condições desta execução, não ao gateway isoladamente.

## 1. Método e decisões anteriores à execução

Usamos um caso fictício de assistência técnica com contexto fechado: permite conferir cálculos, fidelidade às fontes e reconhecimento de informação ausente. Não enviamos dados reais. Planejamos uma resposta final por modelo, com chamadas sequenciais e retentativa limitada de 429, com o mesmo prompt, temperatura 0 e limite de 4.096 tokens. Isso é uma demonstração exploratória, não um benchmark estatístico. Temperatura 0 não garante determinismo.

A seleção prioriza dois modelos de famílias diferentes, Liquid e NVIDIA, **somente se constarem no catálogo atual com sufixo `:free`, preços de entrada/saída zero e suporte aos parâmetros utilizados**. Se indisponíveis, o algoritmo escolhe outras famílias elegíveis e registra os IDs efetivos. Não há fallback para modelos pagos. Qwen e Gemma retornaram HTTP 429 em duas rodadas; por isso, a seleção foi alterada antes desta rodada, preservando as falhas em evidências.

Critérios definidos antes das respostas (0 ou 1 ponto por item):
1. Explica RAG como recuperação de informações seguida de geração apoiada nelas.
2. Calcula 6 × 15 = 90 minutos e respeita o limite de 120 minutos.
3. Restringe a troca a 30 dias com comprovante, citando [D1].
4. Informa que o prazo de reembolso não está documentado, sem inventá-lo.
5. Inclui as três seções pedidas e no máximo 180 palavras.

A pontuação exige leitura humana com trechos justificativos; contagens automáticas apenas auxiliam. Latência e tokens são métricas separadas da qualidade.

A rodada preliminar com 1.200 tokens terminou com `length` nos dois modelos: Liquid não produziu texto final e NVIDIA produziu conteúdo truncado. A rodada final aumenta o limite para 4.096 em ambos, sem alterar o prompt. As rodadas preliminares não entram na pontuação final.

In [1]:
import os, json, time, hashlib, platform, re
from pathlib import Path
from datetime import datetime, timezone
from decimal import Decimal
from importlib.metadata import version
import requests
from openai import OpenAI, APIStatusError, APIConnectionError, APITimeoutError
from IPython.display import display, Markdown

def agora():
    return datetime.now(timezone.utc).isoformat()

PASTA = Path("evidencias") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
PASTA.mkdir(parents=True, exist_ok=False)
def salvar(nome, dados):
    (PASTA / nome).write_text(json.dumps(dados, ensure_ascii=False, indent=2), encoding="utf-8")

ambiente = {"python": platform.python_version(), "pacotes": {x: version(x) for x in ["openai", "requests", "nbformat", "nbclient", "ipykernel"]}}
salvar("ambiente.json", ambiente)
print(json.dumps(ambiente, indent=2))

{
  "python": "3.12.7",
  "pacotes": {
    "openai": "2.54.0",
    "requests": "2.34.2",
    "nbformat": "5.11.1",
    "nbclient": "0.11.0",
    "ipykernel": "7.3.0"
  }
}


## 2. Catálogo e confirmação da gratuidade

Fonte consultada diretamente: https://openrouter.ai/api/v1/models. O recorte salvo contém os preços, capacidades e data da consulta. Preço zero no catálogo não assegura disponibilidade ou cota na conta.

In [2]:
catalogo_ok = False
modelos = []
try:
    resposta = requests.get("https://openrouter.ai/api/v1/models", timeout=30)
    resposta.raise_for_status()
    catalogo = resposta.json()["data"]
    def elegivel(m):
        a, preco = m.get("architecture", {}), m.get("pricing", {})
        return (m["id"].endswith(":free")
                and all(Decimal(str(preco.get(k, "-1"))) == 0 for k in ["prompt", "completion"])
                and "text" in a.get("input_modalities", [])
                and "text" in a.get("output_modalities", [])
                and {"temperature", "max_tokens"}.issubset(m.get("supported_parameters", [])))
    gratis = sorted([m for m in catalogo if elegivel(m)], key=lambda m: m["id"])
    por_id = {m["id"]: m for m in gratis}
    candidatos = [por_id[x] for x in ["liquid/lfm-2.5-2.6b:free", "nvidia/nemotron-3.5-lightning:free"] if x in por_id]
    candidatos += gratis
    familias = set()
    for m in candidatos:
        familia = m["id"].split("/")[0]
        if familia not in familias:
            modelos.append(m)
            familias.add(familia)
        if len(modelos) == 2:
            break
    catalogo_ok = len(modelos) == 2
    campos = ["id", "name", "pricing", "context_length", "architecture", "supported_parameters", "top_provider"]
    registro_catalogo = {"consultado_em_utc": agora(), "fonte": resposta.url, "http_status": resposta.status_code,
                        "elegiveis": [m["id"] for m in gratis],
                        "selecionados": [{k: m.get(k) for k in campos} for m in modelos]}
    salvar("catalogo.json", registro_catalogo)
    print(json.dumps(registro_catalogo, ensure_ascii=False, indent=2))
except (requests.RequestException, ValueError, KeyError) as erro:
    salvar("catalogo_erro.json", {"em_utc": agora(), "tipo": type(erro).__name__})
    print("Falha ao consultar o catálogo. Nenhuma inferência será enviada.")

{
  "consultado_em_utc": "2026-09-23T01:15:23.784122+00:00",
  "fonte": "https://openrouter.ai/api/v1/models",
  "http_status": 200,
  "elegiveis": [
    "cohere/north-mini-code:free",
    "dots-studio/dots-3-note-preview:free",
    "google/gemma-4-26b-a4b-it:free",
    "google/gemma-4-31b-it:free",
    "inclusionai/ling-3.0-flash-fin:free",
    "inclusionai/ling-3.0-flash-sante:free",
    "inclusionai/ling-3.0-flash-vl:free",
    "liquid/lfm-2.5-2.6b:free",
    "nex-agi/nex-n2.5-mini:free",
    "nex-agi/nex-n2.5-pro:free",
    "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    "nvidia/nemotron-3-super-120b-a12b:free",
    "nvidia/nemotron-3-ultra-550b-a55b:free",
    "nvidia/nemotron-3.5-content-safety:free",
    "nvidia/nemotron-3.5-lightning:free",
    "poolside/laguna-s-2.1:free",
    "poolside/laguna-xs-2.1:free",
    "qwen/qwen3.8-27b:free",
    "thinkingmachines/inkling-small:free",
    "thinkingmachines/inkling:free",
    "z-ai/glm-5.2:free"
  ],
  "selecionados": [
    

## 3. Prompt idêntico e gabarito

O gabarito é uma referência elaborada para avaliar o contexto fictício, não uma resposta atribuída a qualquer modelo. Espera-se RAG = recuperar e gerar com apoio em fontes; 90 minutos, dentro de 120; troca em 30 dias com comprovante; prazo de reembolso desconhecido. Todas as condições abaixo serão enviadas na mesma mensagem.

In [3]:
PROMPT = """Responda em português, com no máximo 180 palavras, nas seções RAG, Solução e Limitação.
Explique brevemente o que é RAG. Depois, responda ao caso usando somente os documentos abaixo, citando [D1] e [D2] quando usar suas informações. Não invente políticas ausentes.
[D1] A loja fictícia aceita troca de produtos até 30 dias após a compra, mediante comprovante. O documento não informa prazo de reembolso.
[D2] A equipe tem 120 minutos disponíveis. Cada diagnóstico leva 15 minutos. Há 6 produtos para diagnosticar, atendidos sequencialmente.
Caso: uma compra foi feita há 20 dias e tem comprovante. Ela atende às condições de troca? Quanto tempo levam os 6 diagnósticos e cabem no tempo disponível? Qual é o prazo de reembolso?"""
PARAMETROS = {"temperature": 0, "max_tokens": 4096}
prompt_hash = hashlib.sha256(PROMPT.encode()).hexdigest()
salvar("protocolo.json", {"prompt": PROMPT, "sha256": prompt_hash, "parametros": PARAMETROS,
                         "modelos": [m["id"] for m in modelos], "amostras_por_modelo": 1})
print(PROMPT)
print("SHA-256:", prompt_hash)

Responda em português, com no máximo 180 palavras, nas seções RAG, Solução e Limitação.
Explique brevemente o que é RAG. Depois, responda ao caso usando somente os documentos abaixo, citando [D1] e [D2] quando usar suas informações. Não invente políticas ausentes.
[D1] A loja fictícia aceita troca de produtos até 30 dias após a compra, mediante comprovante. O documento não informa prazo de reembolso.
[D2] A equipe tem 120 minutos disponíveis. Cada diagnóstico leva 15 minutos. Há 6 produtos para diagnosticar, atendidos sequencialmente.
Caso: uma compra foi feita há 20 dias e tem comprovante. Ela atende às condições de troca? Quanto tempo levam os 6 diagnósticos e cabem no tempo disponível? Qual é o prazo de reembolso?
SHA-256: 6a68fef07e144ef13fe8b29b2efffeab1ef5a86a4313352259ae9e7b18912ef4


## 4. Execução e evidências

A chave é lida exclusivamente da variável `OPENROUTER_API_KEY`. Não é impressa, gravada ou inserida no notebook. Cabeçalhos de autenticação e corpos de erro não são registrados. Usamos o SDK OpenAI com o endpoint do OpenRouter, como na aula.

Desabilitamos retentativas automáticas do SDK. Uma célula separada permite uma retentativa explícita por falha 429, respeitando Retry-After de até 60 segundos e preservando o erro original. Timeout de inatividade: 90 segundos; a duração total pode ser maior se o serviço mantiver a conexão ativa. Em HTTP 429, registrar e reexecutar mais tarde conforme a cota/Retry-After; em 401/403, verificar credenciais/permissões; em 402, verificar a conta; em 5xx, aguardar recuperação. Falhas não recebem pontuação de qualidade nem respostas simuladas. Cada reexecução cria uma pasta de evidências separada.

In [4]:
resultados = []
chave = os.environ.get("OPENROUTER_API_KEY", "").strip()
if not chave or not catalogo_ok:
    motivo = "OPENROUTER_API_KEY ausente" if not chave else "Menos de dois modelos elegíveis"
    salvar("pendencia.json", {"em_utc": agora(), "motivo": motivo, "inferencias_realizadas": 0})
    print("PENDENTE:", motivo, "— comparação de respostas ainda não realizada.")
else:
    cliente = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=chave, timeout=90, max_retries=0)
    for modelo in modelos:
        registro = {"modelo_solicitado": modelo["id"], "inicio_utc": agora(), "prompt_sha256": prompt_hash,
                    "parametros": PARAMETROS, "tentativas": 1}
        inicio = time.perf_counter()
        try:
            raw = cliente.chat.completions.with_raw_response.create(
                model=modelo["id"], messages=[{"role": "user", "content": PROMPT}], **PARAMETROS)
            saida = raw.parse()
            dados = saida.model_dump()
            escolha = saida.choices[0] if saida.choices else None
            texto = escolha.message.content if escolha else None
            registro.update({"http_status": raw.status_code, "id_resposta": saida.id,
                             "modelo_retornado": saida.model, "provedor": dados.get("provider"),
                             "finish_reason": escolha.finish_reason if escolha else None,
                             "uso": dados.get("usage"), "resposta": texto,
                             "status": "sucesso" if texto else "sem_texto"})
        except APIStatusError as erro:
            corpo = erro.body if isinstance(erro.body, dict) else {}
            mensagem = str(corpo.get("message", "")).lower()
            categoria = "cota_diaria" if "day" in mensagem or "daily" in mensagem else "limite_ou_capacidade" if erro.status_code == 429 else "outro"
            registro["categoria_erro"] = categoria
            registro.update({"status": "erro_http", "http_status": erro.status_code,
                             "retry_after": erro.response.headers.get("retry-after")})
        except (APITimeoutError, APIConnectionError) as erro:
            registro.update({"status": "erro_conexao", "tipo": type(erro).__name__})
        except Exception as erro:
            registro.update({"status": "erro_local_ou_resposta", "tipo": type(erro).__name__})
        registro["latencia_s"] = round(time.perf_counter() - inicio, 3)
        registro["fim_utc"] = agora()
        resultados.append(registro)
        salvar(f"modelo_{len(resultados)}.json", registro)
        print(json.dumps(registro, ensure_ascii=False, indent=2))
    cliente.close()
del chave
salvar("resultados.json", resultados)

{
  "modelo_solicitado": "liquid/lfm-2.5-2.6b:free",
  "inicio_utc": "2026-09-23T01:15:23.864303+00:00",
  "prompt_sha256": "6a68fef07e144ef13fe8b29b2efffeab1ef5a86a4313352259ae9e7b18912ef4",
  "parametros": {
    "temperature": 0,
    "max_tokens": 4096
  },
  "tentativas": 1,
  "categoria_erro": "limite_ou_capacidade",
  "status": "erro_http",
  "http_status": 429,
  "retry_after": "35",
  "latencia_s": 1.322,
  "fim_utc": "2026-09-23T01:15:25.186307+00:00"
}


{
  "modelo_solicitado": "nvidia/nemotron-3.5-lightning:free",
  "inicio_utc": "2026-09-23T01:15:25.192650+00:00",
  "prompt_sha256": "6a68fef07e144ef13fe8b29b2efffeab1ef5a86a4313352259ae9e7b18912ef4",
  "parametros": {
    "temperature": 0,
    "max_tokens": 4096
  },
  "tentativas": 1,
  "http_status": 200,
  "id_resposta": "gen-1790126125-P8BUmYHArDNRQw6vTsev",
  "modelo_retornado": "nvidia/nemotron-3.5-lightning:free",
  "provedor": "Nvidia",
  "finish_reason": "stop",
  "uso": {
    "completion_tokens": 15,
    "prompt_tokens": 221,
    "total_tokens": 236,
    "completion_tokens_details": {
      "accepted_prediction_tokens": null,
      "audio_tokens": 0,
      "reasoning_tokens": 9,
      "rejected_prediction_tokens": null,
      "image_tokens": 0
    },
    "prompt_tokens_details": {
      "audio_tokens": 0,
      "cache_write_tokens": 0,
      "cached_tokens": 0,
      "video_tokens": 0
    },
    "cost": 0,
    "is_byok": false,
    "cost_details": {
      "upstream_inferenc

In [6]:
# Recuperação controlada: repete apenas falhas HTTP 429; não descarta respostas ruins.
import os, json, time
from pathlib import Path
from datetime import datetime, timezone
from openai import OpenAI, APIStatusError
from IPython.display import display, Markdown
PASTA = sorted(Path("evidencias").iterdir())[-1]
protocolo = json.loads((PASTA / "protocolo.json").read_text())
resultados = json.loads((PASTA / "resultados.json").read_text())
for indice, original in enumerate(list(resultados)):
    if original.get("http_status") != 429 or not os.environ.get("OPENROUTER_API_KEY"):
        continue
    espera = original.get("retry_after") or "35"
    espera = int(espera) if str(espera).isdigit() else 60
    if espera > 60:
        print("Retentativa adiada: intervalo superior a 60 segundos.")
        continue
    print("Aguardando", espera, "segundos para repetir", original["modelo_solicitado"])
    time.sleep(espera)
    r = {"modelo_solicitado": original["modelo_solicitado"], "inicio_utc": datetime.now(timezone.utc).isoformat(),
         "prompt_sha256": protocolo["sha256"], "parametros": protocolo["parametros"], "tentativas": 2,
         "espera_antes_retentativa_s": espera, "registro_anterior": f"modelo_{indice+1}.json"}
    inicio = time.perf_counter()
    try:
        with OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"], timeout=90, max_retries=0) as cliente:
            raw = cliente.chat.completions.with_raw_response.create(model=r["modelo_solicitado"],
                messages=[{"role": "user", "content": protocolo["prompt"]}], **protocolo["parametros"])
            saida = raw.parse()
            dados = saida.model_dump()
            escolha = saida.choices[0] if saida.choices else None
            texto = escolha.message.content if escolha else None
            r.update({"http_status": raw.status_code, "id_resposta": saida.id, "modelo_retornado": saida.model,
                      "provedor": dados.get("provider"), "finish_reason": escolha.finish_reason if escolha else None,
                      "uso": dados.get("usage"), "resposta": texto, "status": "sucesso" if texto else "sem_texto"})
    except APIStatusError as erro:
        r.update({"status": "erro_http", "http_status": erro.status_code, "retry_after": erro.response.headers.get("retry-after")})
    except Exception as erro:
        r.update({"status": "erro", "tipo": type(erro).__name__})
    r["latencia_s"] = round(time.perf_counter()-inicio, 3)
    r["fim_utc"] = datetime.now(timezone.utc).isoformat()
    (PASTA / f"modelo_{indice+1}_retentativa.json").write_text(json.dumps(r, ensure_ascii=False, indent=2))
    resultados[indice] = r
    print(json.dumps(r, ensure_ascii=False, indent=2))
(PASTA / "resultados.json").write_text(json.dumps(resultados, ensure_ascii=False, indent=2))


Aguardando 35 segundos para repetir liquid/lfm-2.5-2.6b:free


{
  "modelo_solicitado": "liquid/lfm-2.5-2.6b:free",
  "inicio_utc": "2026-09-23T01:18:40.496927+00:00",
  "prompt_sha256": "6a68fef07e144ef13fe8b29b2efffeab1ef5a86a4313352259ae9e7b18912ef4",
  "parametros": {
    "temperature": 0,
    "max_tokens": 4096
  },
  "tentativas": 2,
  "espera_antes_retentativa_s": 35,
  "registro_anterior": "modelo_1.json",
  "http_status": 200,
  "id_resposta": "gen-1790126321-2XtGbsImjvpp2z83dWGc",
  "modelo_retornado": "liquid/lfm-2.5-2.6b:free",
  "provedor": "Liquid",
  "finish_reason": "stop",
  "uso": {
    "completion_tokens": 1894,
    "prompt_tokens": 208,
    "total_tokens": 2102,
    "completion_tokens_details": {
      "accepted_prediction_tokens": null,
      "audio_tokens": 0,
      "reasoning_tokens": 1681,
      "rejected_prediction_tokens": null,
      "image_tokens": 0
    },
    "prompt_tokens_details": {
      "audio_tokens": 0,
      "cache_write_tokens": 0,
      "cached_tokens": 0,
      "video_tokens": 0
    },
    "cost": 0,
    "i

3540

## 5. Comparativo observado

A latência mede o tempo total de cada chamada no cliente, incluindo rede e serviço, e não apenas a inferência. Tokens dependem do tokenizador. `finish_reason=length` indica possível truncamento. Provedor ou custo ausente significa “não informado”, não valor zero.

In [7]:
linhas = ["| Modelo | Estado | Latência (s) | Tokens entrada/saída | Palavras | Término |",
          "|---|---|---:|---|---:|---|"]
for r in resultados:
    uso = r.get("uso") or {}
    palavras = len((r.get("resposta") or "").split())
    linhas.append(f"| {r['modelo_solicitado']} | {r['status']} | {r['latencia_s']} | {uso.get('prompt_tokens', 'N/D')}/{uso.get('completion_tokens', 'N/D')} | {palavras} | {r.get('finish_reason', 'N/D')} |")
if resultados:
    display(Markdown("\n".join(linhas)))
else:
    display(Markdown("**Sem respostas de modelos. Não é possível comparar qualidade ou latência de inferência.**"))
for r in resultados:
    print("\nRESPOSTA INTEGRAL:", r['modelo_solicitado'])
    print(r.get("resposta") or "Não houve resposta textual.")

| Modelo | Estado | Latência (s) | Tokens entrada/saída | Palavras | Término |
|---|---|---:|---|---:|---|
| liquid/lfm-2.5-2.6b:free | sucesso | 12.54 | 208/1894 | 127 | stop |
| nvidia/nemotron-3.5-lightning:free | sucesso | 110.281 | 221/15 | 1 | stop |


RESPOSTA INTEGRAL: liquid/lfm-2.5-2.6b:free
**RAG**: RAG (Retrieval‑Augmented Generation) combina busca de informações externas com geração de texto por um modelo de linguagem, permitindo responder com base em dados atualizados.  

**Solução**: A compra ocorreu há 20 dias e possui comprovante, portanto cumpre a regra de troca da loja (até 30 dias com comprovante) conforme [D1]. Os seis diagnósticos levam 6 × 15 min = 90 min, que cabe dentro dos 120 min disponíveis [D2]. Assim, os diagnósticos são concluídos a tempo. Sobre o prazo de reembolso, nenhum documento menciona um limite específico; apenas a condição de troca está descrita, então o prazo de reembolso não pode ser determinado a partir das fontes fornecidas.  

**Limitação**: A informação sobre o prazo de reembolso está ausente nos documentos, impedindo uma resposta definitiva sobre esse ponto.

RESPOSTA INTEGRAL: nvidia/nemotron-3.5-lightning:free
Heressj­x​t​​­s 


## 6. Análise crítica e conclusão

**Experimento concluído com duas respostas reais.** Rodada final: `20260923T011523536232Z` (UTC; 22/09/2026 no fuso de São Paulo). A leitura qualitativa abaixo se refere exclusivamente aos IDs `gen-1790126321-2XtGbsImjvpp2z83dWGc` e `gen-1790126125-P8BUmYHArDNRQw6vTsev`. Se o notebook for reexecutado, é necessário revisar esta análise conforme as novas respostas.

| Critério predefinido | Liquid LFM 2.5 2.6B | NVIDIA Nemotron 3.5 Lightning |
|---|---|---|
| 1. Explicação de RAG | **1** — “combina busca de informações externas com geração de texto” | **0** — não explica RAG |
| 2. Cálculo e capacidade | **1** — informa 6 × 15 = 90 min e compara com 120 min [D2] | **0** — não apresenta cálculo |
| 3. Política de troca | **1** — “até 30 dias com comprovante”, citando [D1], e aplica aos 20 dias | **0** — não responde à pergunta |
| 4. Informação ausente | **1** — “o prazo de reembolso não pode ser determinado” | **0** — não reconhece a ausência de informação |
| 5. Formato e extensão | **1** — três seções e 127 palavras por separação em espaços | **0** — texto ininteligível, sem as três seções |
| **Total** | **5/5** | **0/5** |

A resposta completa da NVIDIA está preservada na seção 5 e no JSON. Ela começa com `Heressj` e contém caracteres invisíveis; não apresenta conteúdo avaliável sobre o caso. A falta de invenção explícita não basta para ganhar o ponto 4: o critério exige reconhecer que a informação não existe. A resposta do Liquid é compreensível, resolve os cálculos, cita as fontes e reconhece a lacuna. Há uma ressalva conceitual: RAG permite consultar fontes externas, mas essas fontes não são necessariamente atualizadas; a frase do Liquid sobre “dados atualizados” simplifica essa condição. Isso não invalida o critério básico de recuperar e gerar.

| Métrica da resposta final | Liquid | NVIDIA |
|---|---:|---:|
| Latência da chamada, em segundos | 12.54 | 110.281 |
| Tokens de entrada | 208 | 221 |
| Tokens de saída, incluindo raciocínio | 1894 | 15 |
| Tokens de raciocínio informados | 1681 | 9 |
| Custo informado pela API, USD | 0 | 0 |
| Provedor informado | Liquid | Nvidia |
| Término | stop | stop |

O Liquid foi mais útil e sua chamada final foi mais rápida nesta amostra. Sua latência não inclui os 35 segundos de espera nem a chamada anterior recusada com 429. Não interpretamos a resposta curta da NVIDIA como eficiência: ela não resolveu a tarefa. `status=sucesso` nos registros indica retorno textual pela API, não aprovação nos critérios de qualidade. `stop` também não garante correção.

### Ocorrências e decisões registradas

1. A primeira execução consultou o catálogo sem chave e registrou a pendência; não simulou inferências.
2. Qwen e Gemma retornaram HTTP 429 em duas rodadas. As quatro falhas estão preservadas.
3. Liquid e NVIDIA foram escolhidos no catálogo como alternativas gratuitas de famílias distintas, antes de observar suas respostas finais.
4. Com 1.200 tokens, Liquid esgotou o limite em raciocínio sem texto final; NVIDIA retornou conteúdo truncado. Esses resultados preliminares não foram pontuados como respostas finais.
5. O limite subiu para 4.096 nos dois modelos. Liquid retornou 429 com `Retry-After: 35`; uma retentativa respeitou esse intervalo e produziu resposta completa. NVIDIA produziu texto ininteligível, que foi mantido e recebeu 0/5. Não houve repetição para substituir essa resposta por uma melhor.

**Conclusão:** para este prompt e esta execução, o Liquid foi a opção adequada. O resultado da NVIDIA demonstra a necessidade de validar conteúdo mesmo após HTTP 200. Não é possível atribuir a falha exclusivamente ao modelo, pois o experimento não isola o provedor ou problemas transitórios. A atividade também mostra que preços gratuitos não asseguram disponibilidade imediata e que tokens de raciocínio podem consumir o orçamento antes da resposta final.

**Limitações:** apenas uma resposta final por modelo; amostragem sequencial em horários diferentes; alteração do limite após o piloto; seleção condicionada à disponibilidade; oscilação de carga; diferentes tokenizadores e treinamento. Temperatura zero não assegura determinismo. Não calculamos médias, significância estatística ou superioridade geral. O contexto fornecido reduz a dificuldade e não testa recuperação real: avaliamos geração apoiada em contexto, sem implementar um pipeline RAG.


## 7. Segurança e referências

Só o caso fictício vai ao gateway e ao provedor. Não incluímos identificação acadêmica, chave, arquivos de ambiente ou respostas de endpoints de conta. Os JSONs preservam apenas os campos necessários ao experimento. Não publicamos corpos de erro, que podem conter detalhes da conta. Antes de enviar, revisar o notebook, os JSONs e o README.

Referências:
- Material da disciplina: *Aula 09.1 — OpenRouter — Gateway Multi-Modelo*, slides 3–11.
- Catálogo ao vivo: https://openrouter.ai/api/v1/models
- Integração: https://openrouter.ai/docs/quickstart
- Limites e erros: https://openrouter.ai/docs/api/reference/limits

As evidências registram a data efetiva desta execução. Não presumimos que ela seja a data da aula.
- Tokens de raciocínio e limite de saída: https://openrouter.ai/docs/guides/best-practices/reasoning-tokens

A análise qualitativa foi elaborada com assistência de IA e deve ser revisada pela dupla antes do envio. Nenhuma resposta foi atribuída aos modelos sem retorno real da API.